# 02c — Cleaning charts and EDA (Rutu)
**Project:** DATA230 Group 4 · **Data:** cleaned FAERS GLP-1 reports from `01_data_cleaning.ipynb`

| Chart | Question | Chart type and why |
|---|---|---|
| **C1** | How did 7.6 million raw FDA reports become our dataset? | Horizontal funnel bars: each step is a count of the step before |
| **C3** | What did age cleaning change? | Bar of what happened to each report's age + cleaned age histogram |
| **Reaction groups** | Which kinds of side effects are reported? | Mapping table, saved to `src/reaction_groups.csv` |
| **E5** | Why was each drug taken, and does the reason relate to serious reports? | 100% stacked bar + % serious with 95% interval |
| **E3** | Which side effects are linked to a report being serious? | Dot plot: % serious per reaction group vs overall rate |

All counts are **reports, not patients**, and all comparisons are **among reported cases**.

## 0 · Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from src.definitions import DRUG_COLORS, GLP1_DRUGS, AGE_TO_YEARS

DATA_DIR = PROJECT_ROOT / "data" / "02_intermediate"
FIG_DIR = PROJECT_ROOT / "figures"; FIG_DIR.mkdir(exist_ok=True)
SUFFIX = ""          # "" = final 2022-2026 files; "_2025" = practice files

reports = pd.read_parquet(DATA_DIR / f"glp1_reports{SUFFIX}.parquet")
reactions = pd.read_parquet(DATA_DIR / f"glp1_reactions{SUFFIX}.parquet")
counts = pd.read_csv(DATA_DIR / f"pipeline_counts{SUFFIX}.csv")
print("reports:", reports.shape, "| reactions:", reactions.shape)

# Same look as the team's other notebooks
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "figure.dpi": 110, "savefig.dpi": 200,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
    "axes.titleweight": "bold", "axes.titlesize": 12,
})
INK, MUTED, GREY = "#222222", "#6B6F76", "#B9BDC5"
ACCENT = "#2B6E9E"                       # single-series colour (not a drug)
DRUG_ORDER = ["tirzepatide", "semaglutide", "dulaglutide", "liraglutide"]   # largest -> smallest
LABEL = {d: d.title() for d in GLP1_DRUGS}
# Generic name + brand names, so a reader without medical background recognises the drug
BRAND_LABEL = {"semaglutide": "Semaglutide\n(Ozempic, Wegovy, Rybelsus)", "tirzepatide": "Tirzepatide\n(Mounjaro, Zepbound)",
               "dulaglutide": "Dulaglutide\n(Trulicity)", "liraglutide": "Liraglutide\n(Victoza, Saxenda)"}

# Plain-English text reused in figure captions (Lecture 2: every figure needs a caption that makes it understandable alone)
SOURCE = ("Data: FDA Adverse Event Reporting System (FAERS) public files, 2022 Q1 - 2026 Q2, reports where a GLP-1 drug "
          "is the main suspected drug. Counts are reports, not patients.")
SERIOUS_DEF = ("'Serious' = the report lists death, a life-threatening event, hospitalisation, disability, "
               "a birth defect or a needed medical intervention.")
CI_DEF = "Whisker = 95% confidence interval (the range the true percentage most likely lies in); n = number of reports."

import textwrap
def caption(fig, text, y=-0.03):
    width = int(fig.get_figwidth() * 15)
    fig.text(0.01, y, textwrap.fill(text, width), ha="left", va="top", fontsize=8.5, color=MUTED)

def savefig(name):
    path = FIG_DIR / f"{name}.png"
    plt.savefig(path, bbox_inches="tight"); print("saved ->", path.name)

def wilson(k, n, z=1.96):
    k = np.asarray(k, float); n = np.asarray(n, float)
    p = k / n; den = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return c - h, c + h

## C1 · From all raw FAERS reports to our GLP-1 dataset
Each bar is one cleaning step. The label shows how many reports were left and what share of the previous step was kept.

In [ ]:
c1 = counts.copy()
c1["kept_pct"] = (c1["reports"] / c1["reports"].shift(1) * 100).round(1)
short = {0: "All FDA side-effect reports\n(every drug, repeats included)",
         1: "One report per case\n(latest update kept, FDA-withdrawn removed)",
         2: "Report mentions a GLP-1 drug",
         3: "GLP-1 is the main suspected drug\n(our dataset)"}
c1["label"] = [short.get(i, s) for i, s in enumerate(c1["step"])]

fig, ax = plt.subplots(figsize=(10, 4))
y = np.arange(len(c1))[::-1]
colors = [GREY] * (len(c1) - 1) + [ACCENT]
ax.barh(y, c1["reports"], color=colors, height=0.6)
xmax = c1["reports"].max()
for yi, (_, r) in zip(y, c1.iterrows()):
    txt = f"{int(r['reports']):,}"
    if not np.isnan(r["kept_pct"]):
        txt += f"   ({r['kept_pct']:.1f}% of previous step)"
    ax.text(r["reports"] + xmax * 0.01, yi, txt, va="center", fontsize=9.5, color=INK)
ax.set_yticks(y); ax.set_yticklabels(c1["label"])
ax.set_xlim(0, xmax * 1.55); ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: "0" if v == 0 else f"{v/1e6:.0f}M"))
ax.set_xlabel("Number of side-effect reports")
final = int(c1["reports"].iloc[-1]); raw = int(c1["reports"].iloc[0])
ax.set_title(f"How did {raw/1e6:.1f} million FDA side-effect reports become our {final:,} GLP-1 reports?")
plt.tight_layout()
caption(fig, "Each bar is one cleaning step; the label shows how many reports remain and what share of the previous step "
             "was kept. FAERS is the FDA's public database of suspected side-effect reports sent by patients, doctors and "
             "drug companies. One case can be re-sent several times as it is updated, so only its latest version is kept. "
             "'Main suspected drug' = the drug the reporter believed caused the problem. " + SOURCE)
savefig("C1_pipeline_funnel"); plt.show()

dropped_versions = int(c1["reports"].iloc[0] - c1["reports"].iloc[1])
print("INSIGHT:")
print(f"  {dropped_versions:,} raw rows were older versions of the same case or FDA-deleted cases.")
print(f"  {int(c1['reports'].iloc[2] - c1['reports'].iloc[3]):,} GLP-1 reports were dropped because the GLP-1 drug was not the main suspect.")
print(f"  Final dataset: {final:,} reports = {final/raw*100:.1f}% of the raw rows.")
print("DECISION: keep only the latest version of each case and only primary-suspect GLP-1 reports,")
print("          so each report is counted once and the GLP-1 drug is the drug under suspicion.")

## C3 · Age: what cleaning did
FAERS stores age as a number **plus a unit** (`age_cod`: years, months, weeks, days, decades, hours).
Left: what happened to the age of every report. Right: the cleaned age distribution.

In [ ]:
unit_counts = reports["age_cod"].fillna("missing").value_counts()
print("Age units in the data:", unit_counts.to_dict())

n_all      = len(reports)
no_age     = int(reports["age_raw"].isna().sum())
usable     = reports["age_years"].notna()
converted  = int((usable & reports["age_cod"].notna() & (reports["age_cod"] != "YR")).sum())
kept_years = int((usable & (reports["age_cod"] == "YR")).sum())
invalid    = int((reports["age_raw"].notna() & ~usable).sum())     # impossible or unknown unit -> set missing
over       = int((reports["age_raw"] > 110).sum())
clean      = reports["age_years"].dropna()
miss       = (~usable).mean() * 100

steps = pd.DataFrame({
    "label": ["Age given in years (kept)", "Age in months/weeks/days/decades\n(converted to years)",
              "Impossible value or unit\n(set to missing)", "No age reported\n(kept as 'Unknown')"],
    "n": [kept_years, converted, invalid, no_age],
    "color": [ACCENT, ACCENT, GREY, GREY]})

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"width_ratios": [1.1, 1]})
a1.barh(steps["label"], steps["n"], color=steps["color"], height=0.6)
a1.invert_yaxis()
for i, n in enumerate(steps["n"]):
    a1.text(n + n_all * 0.01, i, f"{n:,}  ({'<0.1' if 0 < n / n_all * 100 < 0.1 else f'{n / n_all * 100:.1f}'}%)", va="center", fontsize=9, color=INK)
a1.set_xlim(0, steps["n"].max() * 1.45)
a1.set_xlabel("Number of reports")
a1.set_title(f"What happened to age ({n_all:,} reports)")
a1.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e3:.0f}K" if v else "0"))

bins = np.arange(0, 115, 5)
a2.hist(clean, bins=bins, color=ACCENT, edgecolor="white", linewidth=2)
a2.axvline(clean.median(), color=INK, lw=1, ls="--")
a2.set_ylim(0, a2.get_ylim()[1] * 1.12)
a2.text(clean.median() + 1.5, a2.get_ylim()[1] * 0.97, f"median {clean.median():.0f} years", fontsize=9, color=INK, va="top",
        bbox=dict(facecolor="white", edgecolor="none", pad=1))
a2.set_title("Cleaned age (years), reports with a usable age")
a2.set_xlabel("Age (years)"); a2.set_ylabel("Number of reports")
a2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e3:g}K" if v >= 1000 else f"{v:.0f}"))
from matplotlib.patches import Patch
a1.legend(handles=[Patch(color=ACCENT, label="usable age"), Patch(color=GREY, label="no usable age")],
          loc="center right", frameon=False, fontsize=9)
fig.suptitle("How much of the patient-age information is usable after cleaning?", fontweight="bold", y=1.02)
plt.tight_layout()
caption(fig, "FAERS stores age as a number plus a unit (years, months, weeks, days or decades). We converted every age "
             "to years and treated values above 110 as impossible. Reports with no usable age are kept and labelled "
             "'Unknown' instead of being deleted. Right panel: dashed line = median age. " + SOURCE)
savefig("C3_age_before_after"); plt.show()

print("INSIGHT:")
print(f"  {converted:,} reports had age in another unit and were converted; {invalid:,} impossible values were set to missing "
      f"({over:,} raw values were above 110).")
print(f"  {no_age:,} reports ({no_age / n_all * 100:.1f}%) never gave an age. Usable age: {usable.sum():,} reports, median {clean.median():.0f} years.")
print("DECISION: use age_years / age_group only (never age_raw); keep missing age as 'Unknown', impute inside the ML pipeline.")

## Reaction groups
FAERS uses thousands of standard reaction terms (MedDRA "preferred terms"). The official grouping needs a licence,
so we group terms with simple keyword rules, then **check the most common terms by eye**.
The first matching rule wins, so specific groups come before general ones.

In [ ]:
RULES = [
    ("Outcome words (death, hospital)", ["DEATH", "DIED", "FATAL", "HOSPITALI"]),
    ("Injection site",            ["INJECTION SITE"]),
    ("Pancreas & gallbladder",    ["PANCREA", "CHOLE", "GALLBLADDER", "BILIARY", "LIPASE", "AMYLASE"]),
    ("Dosing, device & product use", ["DOSE", "DOSING", "MEDICATION ERROR", "OFF LABEL", "PRODUCT ", "DEVICE",
                                  "WRONG", "INCORRECT", "ACCIDENTAL", "OVERDOSE", "UNDERDOSE", "EXPOSURE",
                                  "STORAGE", "INAPPROPRIATE SCHEDULE", "TECHNIQUE", "PRESCRIBED", "MISUSE",
                                  "COUNTERFEIT", "TITRATION", "THERAPY CESSATION", "THERAPY INTERRUPTED",
                                  "DRUG INTERACTION", "NEEDLE", "PRESCRIPTION", "THERAPY CHANGE"]),
    ("Lack of effect",            ["INEFFECTIVE", "NO THERAPEUTIC", "THERAPEUTIC RESPONSE", "THERAPY NON-RESPONDER",
                                  "LESS THAN EXPECTED", "TREATMENT FAILURE", "INADEQUATE CONTROL"]),
    ("Gastrointestinal",          ["NAUSEA", "VOMIT", "DIARRH", "CONSTIPATION", "ABDOMINAL", "DYSPEPSIA",
                                  "GASTR", "ERUCTATION", "FLATULENCE", "REFLUX", "BOWEL", "ILEUS", "DYSPHAGIA",
                                  "RETCHING", "INTESTIN", "FAECES", "HAEMATOCHEZIA", "BURPING", "COLITIS",
                                  "DIVERTICUL", "APPENDICITIS", "HAEMATEMESIS", "RECTAL", "HICCUP", "DRY MOUTH",
                                  "FOOD POISONING", "OESOPHAG", "COLON"]),
    ("Weight & appetite",         ["WEIGHT", "APPETITE", "HUNGER", "FOOD CRAVING", "EARLY SATIETY", "FEEDING",
                                  "HYPOPHAGIA", "HYPERPHAGIA", "FOOD AVERSION", "EATING DISORDER", "MALNUTRITION",
                                  "FLUID INTAKE"]),
    ("Liver",                     ["HEPAT", "LIVER", "JAUNDICE", "AMINOTRANSFERASE", "TRANSAMINASE", "BILIRUBIN"]),
    ("Cancer & thyroid",          ["NEOPLASM", "CANCER", "CARCINOMA", "MALIGN", "TUMOUR", "LYMPHOMA", "THYROID"]),
    ("Allergy",                   ["HYPERSENSITIVITY", "ANAPHYLA", "ANGIOEDEMA", "SWELLING FACE", "LIP SWELLING",
                                  "ALLERG"]),
    ("Infections & breathing",    ["INFECTION", "COVID", "INFLUENZA", "NASOPHARYNGITIS", "PNEUMONIA", "SEPSIS",
                                  "SINUSITIS", "COUGH", "DYSPNOEA", "ASTHMA", "RHINORRHOEA", "THROAT", "HERPES",
                                  "BRONCH", "RESPIRATORY"]),
    ("Heart & circulation",       ["CARDIAC", "HEART", "PALPITATION", "TACHYCARDIA", "BRADYCARDIA", "BLOOD PRESSURE",
                                  "HYPERTENSION", "HYPOTENSION", "ATRIAL", "MYOCARDIAL", "CEREBROVASCULAR",
                                  "STROKE", "THROMBOSIS", "EMBOLISM", "ARRHYTHMIA", "CHEST", "ISCHAEMIC ATTACK"]),
    ("Eye",                       ["VISION", "VISUAL", "BLINDNESS", "RETIN", "MACULA", "EYE", "OPTIC", "CATARACT",
                                  "LENS", "DIPLOPIA", "FLOATERS"]),
    ("Nervous system",            ["HEADACHE", "DIZZ", "MIGRAINE", "SYNCOPE", "SEIZURE", "PARAESTHESIA",
                                  "TREMOR", "SOMNOLENCE", "NEUROPATHY", "TASTE", "DYSGEUSIA", "AGEUSIA",
                                  "CONSCIOUSNESS", "MEMORY", "AMNESIA", "HYPOAESTHESIA", "BRAIN FOG", "VERTIGO",
                                  "CONFUSION", "BALANCE", "BURNING SENSATION", "ALLODYNIA", "NEURALGIA",
                                  "HYPOACUSIS", "TINNITUS", "DEAFNESS"]),
    ("Mental health & sleep",     ["DEPRESS", "ANXIETY", "SUICID", "INSOMNIA", "MOOD", "PANIC", "EMOTIONAL",
                                  "STRESS", "NERVOUSNESS", "IRRITABILITY", "MENTAL", "SLEEP", "JITTERY"]),
    ("Blood sugar, fluids & kidney", ["GLUCOSE", "GLYCAEMIA", "GLYCEMIA", "GLYCOSYLATED", "DEHYDRATION", "KIDNEY",
                                  "RENAL", "HYPOKALAEMIA", "KETOACIDOSIS", "POTASSIUM", "CREATININE",
                                  "GLOMERULAR", "NEPHRO", "POLLAKIURIA", "THIRST"]),
    ("Muscle, bone & injury",     ["ARTHRALGIA", "MYALGIA", "MUSCLE", "MUSCULAR", "MUSCULOSKELETAL", "ARTHRI",
                                  "ARTHROP", "JOINT", "BACK PAIN", "GAIT", "MOBILITY", "FALL", "INJURY",
                                  "FRACTURE", "CONTUSION", "TRAFFIC"]),
    ("Skin & hair",               ["ALOPECIA", "RASH", "PRURITUS", "URTICARIA", "HAIR", "SKIN", "ERYTHEMA",
                                  "HYPERHIDROSIS"]),
    ("General (tiredness, pain)", ["FATIGUE", "ASTHENIA", "MALAISE", "PAIN", "CHILLS", "FEELING", "PYREXIA",
                                  "INFLUENZA LIKE", "ILLNESS", "LETHARGY", "DISCOMFORT", "SWELLING", "COLD SWEAT",
                                  "HOT FLUSH", "INFLAMMATION", "CONDITION AGGRAVATED", "GENERAL PHYSICAL"]),
]
GROUP_ORDER = [g for g, _ in RULES] + ["Other"]

def group_term(pt):
    t = str(pt).upper()
    for grp, words in RULES:
        if any(w in t for w in words):
            return grp
    return "Other"

terms = (reactions.groupby("pt")["primaryid"].nunique()
         .rename("reports").sort_values(ascending=False).reset_index())
terms["reaction_group"] = terms["pt"].map(group_term)
terms["share_of_reports_%"] = (terms["reports"] / reports["primaryid"].nunique() * 100).round(2)

print(f"{len(terms):,} distinct reaction terms. Top 60 with their group (check these by eye):")
print(terms.head(60).to_string(index=False))

top60_other = terms.head(60).query("reaction_group == 'Other'")
print(f"\nTop-60 terms still in 'Other': {len(top60_other)}  -> add a keyword to RULES if any of these clearly belong to a group")
covered = terms.loc[terms["reaction_group"] != "Other", "reports"].sum() / terms["reports"].sum() * 100
print(f"Share of reaction mentions placed in a named group: {covered:.1f}%")

terms.to_csv(PROJECT_ROOT / "src" / "reaction_groups.csv", index=False)
reactions["reaction_group"] = reactions["pt"].map(group_term)
reactions.to_csv(DATA_DIR / f"glp1_reactions_grouped{SUFFIX}.csv", index=False)
print("saved -> src/reaction_groups.csv and data/02_intermediate/glp1_reactions_grouped" + SUFFIX + ".csv (for Tableau)")

## E5 · Why was the drug taken, and does that relate to serious reports?
**Question:** *Among reported cases, do diabetes and weight-loss reports differ by drug and in how often they are serious?*
Left: 100% stacked bar (composition). Right: % serious per indication group with a 95% interval and the number of reports.

In [ ]:
IND_ORDER = ["Diabetes", "Weight loss", "Other", "Unknown"]
# Colours chosen to stay distinct in grayscale (dark / medium / light / light+hatch), Lecture 2 rule
IND_COLORS = {"Diabetes": "#1F4E79", "Weight loss": "#E07B39", "Other": "#B9BDC5", "Unknown": "#EEF0F2"}
IND_HATCH  = {"Unknown": "///"}
IND_NAME   = {"Diabetes": "Diabetes", "Weight loss": "Weight loss", "Other": "Other reason", "Unknown": "Not stated"}
IND_TEXT   = {"Diabetes": "white", "Weight loss": "white", "Other": INK, "Unknown": INK}

mix = (pd.crosstab(reports["drug"], reports["indication_group"])
       .reindex(index=DRUG_ORDER, columns=IND_ORDER, fill_value=0))
mix_pct = mix.div(mix.sum(axis=1), axis=0) * 100

ser = reports.groupby("indication_group")["is_serious"].agg(n="size", k="sum").reindex(IND_ORDER).dropna()
ser["pct"] = ser["k"] / ser["n"] * 100
lo, hi = wilson(ser["k"], ser["n"]); ser["lo"], ser["hi"] = lo * 100, hi * 100

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.3), gridspec_kw={"width_ratios": [1.5, 1]})
ylab = [BRAND_LABEL[d] for d in DRUG_ORDER][::-1]
left = np.zeros(len(DRUG_ORDER))
for ind in IND_ORDER:
    vals = mix_pct[ind].values[::-1]
    a1.barh(ylab, vals, left=left, color=IND_COLORS[ind], label=IND_NAME[ind], hatch=IND_HATCH.get(ind),
            edgecolor="#9AA0A8" if ind in IND_HATCH else "white", linewidth=0.8 if ind in IND_HATCH else 2)
    for yi, (l, w) in enumerate(zip(left, vals)):
        if w >= 6:
            a1.text(l + w / 2, yi, f"{w:.0f}%", ha="center", va="center", color=IND_TEXT[ind], fontsize=9, fontweight="bold",
                    bbox=dict(facecolor=IND_COLORS[ind], edgecolor="none", pad=1) if ind in IND_HATCH else None)
    left += vals
for yi, d in enumerate(DRUG_ORDER[::-1]):
    a1.text(101, yi, f"n={int(mix.loc[d].sum()):,}", va="center", fontsize=9, color=INK)
a1.set_xlim(0, 115); a1.grid(False)
a1.set_xlabel("Share of that drug's reports (%)")
a1.set_title("1 · Reason the drug was taken, as written in the report")
a1.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.45, -0.18), frameon=False, fontsize=9)

ys = np.arange(len(ser))[::-1]
a2.barh(ys, ser["pct"], color=[IND_COLORS[i] for i in ser.index], height=0.6,
        hatch=None, edgecolor=["#9AA0A8" if i in IND_HATCH else "none" for i in ser.index],
        xerr=[ser["pct"] - ser["lo"], ser["hi"] - ser["pct"]], capsize=4, ecolor="#333")
for patch, ind in zip(a2.patches, ser.index):
    if ind in IND_HATCH: patch.set_hatch(IND_HATCH[ind])
for yi, (ind, r) in zip(ys, ser.iterrows()):
    a2.text(r["hi"] + 0.6, yi, f"{r['pct']:.1f}%  (n={int(r['n']):,})", va="center", fontsize=9, color=INK)
a2.set_yticks(ys); a2.set_yticklabels([IND_NAME[i] for i in ser.index]); a2.grid(axis="y", visible=False)
a2.set_xlim(0, ser["hi"].max() * 1.6)
a2.set_xlabel("% of reports that are serious")
a2.set_title("2 · % serious, by reason (all four drugs together)")
fig.suptitle("Why were people taking each GLP-1 drug, and are diabetes reports more often serious?",
             fontweight="bold", y=1.03)
plt.tight_layout()
caption(fig, "Left: each bar is 100% of one drug's reports, split by the reason written in the report. Right: share of "
             "reports that are serious for each reason, all drugs together (Figure E5b checks the difference within "
             "each drug). " + SERIOUS_DEF + " " + CI_DEF + " " + SOURCE, y=-0.04)
savefig("E5_indication"); plt.show()

display(mix_pct.round(1)); display(ser.round(1))
print("INSIGHT (re-read before pasting into a slide):")
for d in DRUG_ORDER:
    print(f"  {LABEL[d]}: diabetes {mix_pct.loc[d,'Diabetes']:.0f}%, weight loss {mix_pct.loc[d,'Weight loss']:.0f}%, unknown {mix_pct.loc[d,'Unknown']:.0f}%")
for ind, r in ser.iterrows():
    print(f"  % serious, {ind}: {r['pct']:.1f}% (n={int(r['n']):,})")
print(f"  Indication is unknown for {mix_pct['Unknown'].mean():.0f}% of reports on average, so any indication result covers only reports that state one.")
print("DECISION: keep indication_group as a model feature with 'Unknown' as its own category; describe results as 'among reports that state an indication'.")

## E5b · Does the diabetes vs weight-loss gap hold once we hold something constant?
Lecture 5 (multivariate act) and Lecture 1 (the satisfaction-by-channel example): a pooled gap can shrink or even
reverse inside every group. Diabetes and weight-loss reports differ in **drug**, **age** and **who reported them**,
so we compare % serious for the two indications **inside each drug, each age group and each reporter type**.
Age groups shown: 18-39, 40-64, 65 and over (under-18 and unknown age left out: too few or no age). "Consumer / other" = every reporter who is not a health professional (consumers, lawyers, missing).

In [ ]:
pair = reports[reports["indication_group"].isin(["Diabetes", "Weight loss"])].copy()
pair["Reporter"] = np.where(pair["is_hcp"] == 1, "Health professional", "Consumer / other")
strata = [("Drug", "drug", DRUG_ORDER),
          ("Age group", "age_group", ["18-39", "40-64", "65 and over"]),
          ("Reporter", "Reporter", ["Health professional", "Consumer / other"])]
rows = []
for block, col, levels in strata:
    for lev in levels:
        sub = pair[pair[col] == lev]
        for ind in ["Diabetes", "Weight loss"]:
            s = sub[sub["indication_group"] == ind]["is_serious"]
            if len(s) >= 30:
                lo, hi = wilson(s.sum(), len(s))
                rows.append({"block": block, "level": lev, "ind": ind, "n": len(s),
                             "pct": s.mean() * 100, "lo": lo * 100, "hi": hi * 100})
st = pd.DataFrame(rows)

MARK = {"Diabetes": "o", "Weight loss": "s"}          # shape + colour, never colour alone (Lecture 2)
PANEL_TITLE = {"Drug": "Within each drug", "Age group": "Within each age group", "Reporter": "By who filed the report"}
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4), gridspec_kw={"wspace": 0.55})
for ax, (block, _, _) in zip(axes, strata):
    d = st[st["block"] == block]
    levels = list(dict.fromkeys(d["level"]))
    for i, lev in enumerate(levels[::-1]):
        for ind, off in [("Diabetes", 0.12), ("Weight loss", -0.12)]:
            r = d[(d["level"] == lev) & (d["ind"] == ind)]
            if r.empty: continue
            r = r.iloc[0]
            ax.plot([r["lo"], r["hi"]], [i + off] * 2, color=IND_COLORS[ind], lw=2)
            ax.scatter(r["pct"], i + off, color=IND_COLORS[ind], marker=MARK[ind], s=50, zorder=3, edgecolor="white")
    ax.set_yticks(range(len(levels)))
    ax.set_yticklabels([BRAND_LABEL.get(l, l) for l in levels[::-1]], fontsize=9)
    ax.grid(axis="y", visible=False)
    ax.set_title(PANEL_TITLE[block])
    ax.set_xlabel("% of reports that are serious")
    ax.set_xlim(0, st["hi"].max() * 1.08)
for ind in ["Diabetes", "Weight loss"]:
    axes[0].scatter([], [], color=IND_COLORS[ind], marker=MARK[ind], label=f"Drug taken for {ind.lower()}")
fig.legend(loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.06))
fig.suptitle("Is the diabetes vs weight-loss difference still there when we compare like with like?",
             fontweight="bold", y=1.02)
plt.tight_layout()
caption(fig, "Each pair of points compares diabetes and weight-loss reports inside one group only (same drug, same age "
             "band or same type of reporter), so the difference cannot come from mixing groups. Lines = 95% confidence "
             "interval. Health professional = doctor, pharmacist or other clinician; 'Consumer / other' = patients, lawyers "
             "and reports with no reporter type. Under-18 and unknown-age reports are not shown. " + SERIOUS_DEF, y=-0.1)
savefig("E5b_indication_stratified"); plt.show()

wide = st.pivot_table(index=["block", "level"], columns="ind", values=["pct", "lo", "hi", "n"])
display(wide.round(1))
both = wide.dropna()
higher = both[both[("lo", "Diabetes")] > both[("hi", "Weight loss")]]
lower  = both[both[("hi", "Diabetes")] < both[("lo", "Weight loss")]]
print("INSIGHT (re-read before pasting into a slide):")
print(f"  Pooled: diabetes {ser.loc['Diabetes','pct']:.1f}% vs weight loss {ser.loc['Weight loss','pct']:.1f}% serious.")
print(f"  Inside {len(both)} groups: diabetes clearly higher in {len(higher)}, clearly lower in {len(lower)}, "
      f"overlapping in {len(both) - len(higher) - len(lower)}.")
for (block, lev), r in both.iterrows():
    print(f"    {block:9s} {LABEL.get(lev, lev):20s} diabetes {r[('pct','Diabetes')]:5.1f}%  vs  weight loss {r[('pct','Weight loss')]:5.1f}%")
print("DECISION: if the gap holds inside most groups, indication is a useful model feature on its own;")
print("          if it shrinks or flips, the pooled gap came from drug/age/reporter mix -> keep those as features together.")

## E3 · Which side effects are linked to a report being serious?
**Question:** *Among reported cases, how often is a report serious when it mentions each reaction group?*
Dot = % serious among reports that mention at least one reaction in the group; whisker = 95% interval; dashed line = overall % serious.
Groups to the right of the line are **linked to** serious reports (association, not cause).
The "Outcome words" group is greyed out: terms like *death* or *hospitalisation* describe the outcome itself, so they can never be model features (leakage).

In [ ]:
per_report = reactions[["primaryid", "reaction_group"]].drop_duplicates()
per_report = per_report.merge(reports[["primaryid", "is_serious"]], on="primaryid", how="inner")
base = reports["is_serious"].mean() * 100

g = per_report.groupby("reaction_group")["is_serious"].agg(n="size", k="sum")
g = g[g["n"] >= 30]                                   # too few reports -> no stable percentage
g["pct"] = g["k"] / g["n"] * 100
lo, hi = wilson(g["k"], g["n"]); g["lo"], g["hi"] = lo * 100, hi * 100
g = g.sort_values("pct")

LEAK = [x for x in g.index if x.startswith("Outcome words")]
# Plain-English names for the y-axis (the data keeps the short group names)
SHOW = {"Dosing, device & product use": "Wrong dose, pen/device or product problem",
        "Lack of effect": "Drug did not work",
        "Outcome words (death, hospital)": "Words describing the outcome (death, hospitalisation)",
        "Other": "Other / not classified",
        "General (tiredness, pain)": "General (tiredness, pain, fever)",
        "Blood sugar, fluids & kidney": "Blood sugar, dehydration & kidney",
        "Pancreas & gallbladder": "Pancreas & gallbladder (e.g. pancreatitis, gallstones)",
        "Gastrointestinal": "Stomach & gut (nausea, vomiting, diarrhoea)"}
fig, ax = plt.subplots(figsize=(10, 0.42 * len(g) + 1.6))
ys = np.arange(len(g))
for y, (grp, r) in zip(ys, g.iterrows()):
    c = GREY if grp in LEAK + ["Other"] else (ACCENT if r["lo"] > base else "#8A8F98")
    ax.plot([r["lo"], r["hi"]], [y, y], color=c, lw=2, solid_capstyle="round")
    ax.scatter(r["pct"], y, s=70, color=c, zorder=3, edgecolor="white", linewidth=1.5)
    lab = f"{r['pct']:.1f}%  (n={int(r['n']):,})"
    if grp in LEAK and r["pct"] > 50:
        ax.text(r["lo"] - 1.5, y, "describes the outcome itself, not used in the model →  " + lab, va="center", ha="right", fontsize=9, color=INK)
    else:
        ax.text(r["hi"] + 1.5, y, lab + ("  ← describes the outcome itself, not used in the model" if grp in LEAK else ""),
                va="center", fontsize=9, color=INK, bbox=dict(facecolor="white", edgecolor="none", pad=0.5))
ax.axvline(base, color=INK, ls="--", lw=1)
ax.text(base, len(g) - 0.3, f" all GLP-1 reports: {base:.1f}% serious", fontsize=9, color=INK, va="bottom")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=ACCENT, label="clearly more often serious than average"),
                   Line2D([], [], marker="o", ls="", color="#8A8F98", label="not clearly above average"),
                   Line2D([], [], marker="o", ls="", color=GREY, label="not interpreted (outcome words / mixed group)")],
          loc="lower right", frameon=False, fontsize=8.5)
ax.set_yticks(ys); ax.set_yticklabels([SHOW.get(x, x) for x in g.index]); ax.grid(axis="y", visible=False)
ax.set_xlim(0, 100); ax.set_xticks(range(0, 101, 20)); ax.set_ylim(-0.6, len(g) + 0.2)
ax.set_xlabel("% of these reports that are serious")
ax.set_title("When a report lists a given type of side effect, how often is that report serious?")
plt.tight_layout()
caption(fig, "Read each row as: of all reports that list at least one side effect of this type, X% were serious. "
             "One report can list several types, so it can appear in several rows. The 6,411 FDA side-effect terms were "
             "grouped into these types with keyword rules. This shows association, not cause. " + SERIOUS_DEF + " "
             + CI_DEF, y=-0.01)
savefig("E3_serious_by_reaction_group"); plt.show()

display(g.round(1))
named = g.drop(index=LEAK + [x for x in ["Other"] if x in g.index])
above = named[named["lo"] > base].sort_values("pct", ascending=False)
below = named[named["hi"] < base].sort_values("pct")
print("INSIGHT (re-read before pasting into a slide):")
print(f"  Overall, {base:.1f}% of GLP-1 reports are serious.")
for grp, r in above.head(3).iterrows():
    print(f"  Reports mentioning {grp}: {r['pct']:.1f}% serious ({r['pct']/base:.1f}x the overall rate, n={int(r['n']):,})")
for grp, r in below.head(2).iterrows():
    print(f"  Reports mentioning {grp}: only {r['pct']:.1f}% serious (n={int(r['n']):,})")
print("DECISION: reaction groups become yes/no model features for predicting is_serious;")
print("          'Outcome words' terms are removed before modelling (they restate the target -> leakage).")

## Decision table (the output of this EDA, Lecture 5)
One row per issue: what we decided and the number that justifies it. This table goes on a slide and into the report.

In [ ]:
top3 = g.drop(index=[x for x in LEAK + ["Other"] if x in g.index]).sort_values("pct", ascending=False).head(3)
decisions = pd.DataFrame([
    ["Older case versions / FDA-deleted cases", "keep latest version per case; drop deleted",
     f"{dropped_versions:,} raw rows were duplicates or deleted"],
    ["GLP-1 not the primary suspect", "drop the report",
     f"{int(c1['reports'].iloc[2] - c1['reports'].iloc[3]):,} reports; the drug must be the one under suspicion"],
    ["Age in months/weeks/days/decades", "convert to years",
     f"{converted:,} reports ({converted / n_all * 100:.2f}%)"],
    ["Impossible age (> 110)", "set to missing", f"{invalid:,} reports"],
    ["Age not reported", "keep as 'Unknown'; impute inside the ML pipeline",
     f"{no_age:,} reports ({no_age / n_all * 100:.1f}%) - too many to drop"],
    ["Indication not reported", "keep 'Unknown' as its own category",
     f"{(reports['indication_group'] == 'Unknown').mean() * 100:.1f}% of reports"],
    ["6,411 raw reaction terms", "map to reaction groups (keyword rules)",
     f"{covered:.1f}% of reaction mentions placed in a named group"],
    ["Reaction terms that describe the outcome (death, hospitalisation)", "remove before modelling",
     f"{g.loc[LEAK[0], 'pct']:.0f}% serious -> they restate the target (leakage)" if LEAK else "n/a"],
    ["Target is_serious", "classification with class weights / resampling; judge by recall and PR-AUC",
     f"only {base:.1f}% of reports are serious (class imbalance)"],
    ["Reaction groups", "yes/no model features",
     "; ".join(f"{k} {r['pct']:.0f}% serious" for k, r in top3.iterrows()) + f" vs {base:.1f}% overall"],
], columns=["Column / issue", "Decision", "Because"])
with pd.option_context("display.max_colwidth", 120):
    display(decisions.set_index("Column / issue"))

### E3b (backup, not a slide) · Reaction groups by drug
Kept for Q&A and for Brian's dashboard: % of each drug's reports that mention each group.

In [ ]:
per_drug = reactions[["primaryid", "drug", "reaction_group"]].drop_duplicates()
n_reports = reports.groupby("drug")["primaryid"].nunique().reindex(DRUG_ORDER)
heat = (per_drug.groupby(["reaction_group", "drug"])["primaryid"].nunique()
        .unstack("drug").reindex(columns=DRUG_ORDER).fillna(0))
heat = heat.div(n_reports, axis=1) * 100
order = [x for x in heat.mean(axis=1).sort_values(ascending=False).index if x != "Other"]
heat = heat.loc[order + (["Other"] if "Other" in heat.index else [])]
display(heat.round(1))